AI VISION LAB · 05 / STUDENT WORKBOOK

# Mission 01. 데이터 증강 레이어 만들기

이미지 데이터 증강과 재현 가능한 실험



**학습 목표**

- 데이터 증강의 개념과 목적을 설명할 수 있다.
- `RandomRotation`을 사용하여 회전 증강 레이어를 생성할 수 있다.
- 회전 범위와 이미지 보간 방식을 설정할 수 있다.

### VS Code 실행 준비

TensorFlow가 설치된 Python 커널에서 회전 증강 레이어를 생성합니다.

커널 목록에서 TensorFlow가 설치된 Python 환경을 선택하고 위에서부터 실행하세요.

In [1]:
from tensorflow import keras
from tensorflow.keras import layers

## 1. 데이터 증강이란?

**데이터 증강(Data Augmentation)**은 기존 학습 데이터에 변형을 적용하여 다양한 학습 사례를 만드는 기법이다.

이미지를 학습하는 CNN은 학습 데이터에서 이미지의 특징을 찾아낸다. 그러나 학습 이미지가 특정 방향이나 위치에 치우쳐 있다면, 다른 방향이나 위치의 이미지에서는 예측 성능이 떨어질 수 있다.

예를 들어 세로 방향의 Scratch가 포함된 웨이퍼맵을 학습한 CNN이 가로 방향의 Scratch도 잘 분류할 수 있을까?

CNN은 이미지가 회전하더라도 반드시 같은 클래스로 분류하는 것은 아니다. 따라서 학습 이미지에 다양한 방향의 Scratch를 포함시키면 새로운 방향의 패턴을 인식하는 데 도움이 될 가능성이 있다.

하지만 **데이터 증강이 항상 모델 성능을 개선하는 것은 아니다.** 증강으로 만들어진 이미지가 실제 데이터의 특성과 맞지 않거나, 클래스 사이의 차이를 구분하기 어렵게 만들 수도 있다.

이번 실습에서는 회전 증강을 적용해보고 실제로 모델 성능에 어떤 변화가 발생하는지 확인한다.

## 2. 이미지 증강에는 어떤 방법이 있을까?

| 기법 | Keras 레이어 | 이미지 변형 |
| --- | --- | --- |
| 회전 | `RandomRotation` | 무작위 각도로 회전 |
| 반전 | `RandomFlip` | 좌우 또는 상하 반전 |
| 이동 | `RandomTranslation` | 상하좌우로 위치 이동 |
| 확대·축소 | `RandomZoom` | 이미지 크기 변경 |

모두 Keras가 제공하는 데이터 증강 레이어이며, CNN 모델에 추가하거나 별도로 실행할 수 있다. 이번 실습에서는 `RandomRotation`을 사용한다. 다른 증강 레이어도 유사한 방식으로 활용할 수 있다.

## 3. RandomRotation 사용하기
`RandomRotation`은 지정한 범위 안에서 이미지를 무작위로 회전시키는 레이어이다. 다음 코드를 작성하여 회전 증강 레이어를 생성하자.

In [2]:
# 회전 증강 레이어 생성
data_augmentation = keras.Sequential([
    layers.RandomRotation(
        factor=0.25,
        fill_mode="constant",
        fill_value=0.0,
        interpolation="nearest"
    )
])

`keras.Sequential`은 여러 레이어를 순서대로 실행하도록 묶는 객체이다. 현재는 `RandomRotation` 레이어 하나만 포함하고 있다.

| 파라미터 | 설정값 | 의미 |
| --- | --- | --- |
| `factor` | `0.25` | −90° ~ +90° 사이에서 무작위 회전 |
| `fill_mode` | `"constant"` | 회전 후 빈 공간을 일정한 값으로 채움 |
| `fill_value` | `0.0` | 빈 공간을 검은색으로 채움 |
| `interpolation` | `"nearest"` | 가장 가까운 픽셀값을 사용 |

`factor`는 각도를 직접 입력하는 값이 아니다. 전체 360도를 기준으로 회전 범위를 지정한다. 예를 들어 `factor=0.25`는 최대 ±90°의 무작위 회전을 의미한다.

## 4. 이미지 보간(Interpolation)

이미지를 회전시키면 원래 픽셀들이 새로운 위치로 이동한다. 이 과정에서 새로운 위치의 픽셀값을 결정해야 한다. 대표적인 보간 방법에는 `bilinear`와 `nearest`가 있다.

**Bilinear**

주변 픽셀값을 혼합하므로 중간 밝기 값이 생기고 경계가 부드럽게 보일 수 있다.

**Nearest**

가장 가까운 픽셀값을 그대로 사용하므로 중간값이 생기지 않지만 경계가 계단처럼 보일 수 있다.

그림은 보간 원리를 단순화한 개념도이다. 웨이퍼맵에서는 픽셀값이 배경, 정상 다이, 불량 다이 등의 정보를 나타내므로 `nearest`가 적절할 수 있다. 다만 모든 이미지에서 가장 좋은 보간 방법이라는 뜻은 아니다.

## 5. 코드 확인 활동

작성한 코드를 다시 살펴보고 다음 질문에 답해보자.

**Q1.1.**

`factor=0.25`를 `factor=0.125`로 변경하면 이미지의 회전 범위는 어떻게 달라지는가?

**답변**

회전 범위가 **−90°~+90°에서 −45°~+45°**로 줄어듭니다. `factor`에 360°를 곱하면 최대 회전 각도가 됩니다.

**Q1.2.**

현재 웨이퍼맵의 배경은 검은색이다. 회전 후에도 배경을 검은색으로 유지하기 위해 설정한 파라미터 두 가지는 무엇인가?

**답변**

`fill_mode="constant"`로 빈 영역을 일정한 값으로 채우고, `fill_value=0.0`으로 검은색을 지정합니다.

**Q1.3.**

`interpolation="nearest"`를 `"bilinear"`로 변경하면 회전된 이미지의 픽셀값은 어떤 차이가 발생할 수 있는가?

**답변**

`bilinear`는 주변 픽셀을 섞어 새로운 중간값을 만들 수 있습니다. `nearest`는 가장 가까운 픽셀값을 그대로 사용해 경계가 계단처럼 보일 수 있습니다.

**Q1.4.**

현재 `data_augmentation`은 Sequential 객체로 만들었다. 이 객체 안에는 어떤 레이어가 포함되어 있는가?

**답변**

회전 범위가 **±90°**인 `RandomRotation` 레이어 하나가 들어 있습니다.